# Generacion con el modelo base sin afinar (control), Llama 3.1 8B en Ollama

Control experimental del afinamiento con RAFT. Usa **el mismo modelo, Llama 3.1 8B Instruct, sin afinar**, con la misma cuantizacion (GGUF Q4_K_M) y el mismo motor (Ollama), y lo pone a responder exactamente lo mismo que respondio el modelo RAFT. Es la comparacion central del paper (Zhang et al., 2024, Tabla 1: *LLaMA2-7B + RAG* frente a *RAFT*): solo si el modelo afinado supera a su propio modelo base se puede atribuir la diferencia al afinamiento.

**Todo es igual a `generacion_llm_raft.ipynb`, salvo el modelo:**
- las mismas **196 preguntas** y el mismo filtro;
- los mismos contextos ya recuperados por `openai_cs800`;
- los mismos 3 estilos de prompt (`raft`, `estructurado` y `directo`), con los mismos textos;
- `temperature = 0`, `seed = 42`, `num_ctx = 8192`, maximo 1024 tokens de respuesta, y reintentos sin cambiar de modelo.

**No modifica nada existente.** Sus salidas son archivos nuevos, con el mismo formato:
- `./validacion/checkpoints_llm/openai_cs800__llama3.1-8b-base__<prompt>.json`
- `./validacion/resultados_generacion_llm_base.json`: mismo formato que `resultados_generacion_llm_raft.json`.

Despues se evalua con `evaluacion_generacion_ragas_base.ipynb`, y `analisis_raft.ipynb` agrega solo la comparacion RAFT frente a base.

## 0. Descargar el modelo base en Ollama (una sola vez)

Con Ollama abierto, ejecuta en una terminal:
```
ollama pull llama3.1:8b-instruct-q4_K_M
ollama list
```
La descarga pesa unos 4.9 GB. Es la misma cuantizacion Q4_K_M del modelo RAFT, asi que la comparacion no mezcla el efecto de la cuantizacion. Las variables `OLLAMA_FLASH_ATTENTION` y `OLLAMA_KV_CACHE_TYPE` que ya configuraste aplican igual.

In [ ]:
%pip install -q pandas httpx

## 1. Configuracion

In [1]:
import json
import re
import time
from pathlib import Path

import httpx
import pandas as pd

OLLAMA_URL = "http://localhost:11434"

# Modelos locales a evaluar. La clave es el nombre que queda en "modelo_llm" (y en los nombres de archivo).
MODELOS_LOCALES = {
    # El MISMO modelo base del afinamiento, sin afinar, con la misma cuantizacion (Q4_K_M)
    "llama3.1-8b-base": {
        "ollama_id": "llama3.1:8b-instruct-q4_K_M",
        "rol": "base_sin_afinar",
    },
}

# Los mismos 3 estilos que se usaron con el modelo RAFT
ESTILOS_A_CORRER = ["raft", "estructurado", "directo"]

# Los prompts de este flujo (5 fragmentos de ~800 tokens) llegan a ~3-6k tokens; 8192 cubre todos con margen.
# Ollama RECORTA EN SILENCIO lo que exceda num_ctx, por eso se verifica en cada respuesta.
NUM_CTX = 8192
MAX_TOKENS_RESPUESTA = 1024
SEMILLA = 42

# None = las 196 preguntas filtradas. Pon un numero chico (ej. 5) para una prueba rapida.
# OJO: si corres con muestra, borra los checkpoints *__llama3.1-8b-base__* antes de la corrida completa.
TAMANO_MUESTRA_LLM = None

RUTA_CHECKPOINTS_LLM = Path("./validacion/checkpoints_llm")
RUTA_CHECKPOINTS_LLM.mkdir(parents=True, exist_ok=True)
RUTA_SALIDA_LLM = "./validacion/resultados_generacion_llm_base.json"

## 2. Verificar que Ollama y el modelo estan disponibles

In [2]:
_http = httpx.Client(base_url=OLLAMA_URL, timeout=httpx.Timeout(900.0, connect=10.0))

try:
    modelos_instalados = [m["name"] for m in _http.get("/api/tags").json()["models"]]
except Exception as e:
    raise RuntimeError(f"No pude conectar con Ollama en {OLLAMA_URL}. ¿Esta abierto? ({e})")

print("Modelos en Ollama:", modelos_instalados)
for nombre, info in MODELOS_LOCALES.items():
    ok = info["ollama_id"] in modelos_instalados
    print(f"  {nombre:22s} -> {info['ollama_id']:32s} {'OK' if ok else 'NO ENCONTRADO'}")
    if not ok:
        raise RuntimeError(f"Falta el modelo '{info['ollama_id']}' en Ollama: ejecuta `ollama pull {info['ollama_id']}`.")

Modelos en Ollama: ['llama3.1:8b-instruct-q4_K_M', 'raft-acreditacion:latest']
  llama3.1-8b-base       -> llama3.1:8b-instruct-q4_K_M      OK


## 3. Mismas preguntas y mismos contextos que `generacion_llm.ipynb`

El filtro es identico al original: la interseccion de preguntas con exito en ambos pipelines, 196 preguntas. Asi las respuestas son comparables 1 a 1 con las combinaciones `openai_cs800` ya generadas. Para generar solo se usan los contextos de `openai_cs800`.

In [3]:
RUTA_EVALUACION_DETALLE = "./validacion/evaluacion_ragas_detalle.csv"
RUTA_RETRIEVAL_OPENAI = "./validacion/resultados_retrieval.json"

ETIQUETA_OPENAI = "openai/text-embedding-3-large_cs800"
ETIQUETA_BGE = "BAAI/bge-m3_cs800_hibrido"

evaluacion = pd.read_csv(RUTA_EVALUACION_DETALLE)


def preguntas_con_score_positivo(etiqueta):
    sub = evaluacion[evaluacion["configuracion"] == etiqueta]
    validas = sub[(sub["llm_context_precision_with_reference"] > 0) & (sub["context_recall"] > 0)]
    return set(validas["user_input"])


preguntas_validas = preguntas_con_score_positivo(ETIQUETA_OPENAI) & preguntas_con_score_positivo(ETIQUETA_BGE)


def cargar_resultados_config(ruta, etiqueta_config=None):
    with open(ruta, "r", encoding="utf-8") as f:
        contenido = json.load(f)
    if "configuraciones" in contenido:
        for c in contenido["configuraciones"]:
            if f"{c['modelo_embedding']}_cs{c['chunk_size']}" == etiqueta_config:
                return c["resultados"]
        raise ValueError(f"No se encontro la configuracion '{etiqueta_config}' en {ruta}")
    return contenido["resultados"]


resultados_openai = sorted(
    (r for r in cargar_resultados_config(RUTA_RETRIEVAL_OPENAI, ETIQUETA_OPENAI) if r["pregunta"] in preguntas_validas),
    key=lambda r: r["pregunta"],
)

# Solo la mejor configuracion de retrieval
PIPELINES_RETRIEVE = {
    "openai_cs800": resultados_openai,
}
print(f"{len(resultados_openai)} preguntas (deben ser 196, igual que en generacion_llm.ipynb)")

196 preguntas (deben ser 196, igual que en generacion_llm.ipynb)


## 4. Estilos de prompt

Son copia textual de los de `generacion_llm_raft.ipynb`. `directo` y `estructurado` vienen de `generacion_llm.ipynb`, y `raft` de `raft_formato.py`. Para el modelo base, el estilo `raft` es simplemente otro prompt con instrucciones: pide citar entre `##begin_quote##` y responder despues de `##Answer:`, pero el modelo nunca se entreno con ese formato.

In [4]:
def formatear_contexto(contextos):
    return "\n\n".join(f"--- Fragmento {i + 1} ---\n{c}" for i, c in enumerate(contextos))


def prompt_directo(pregunta, contextos):
    return f'''Responde la siguiente pregunta usando UNICAMENTE la informacion del contexto. Si el contexto no contiene la respuesta, dilo explicitamente en vez de inventar o completar con conocimiento externo.

Contexto:
{formatear_contexto(contextos)}

Pregunta: {pregunta}

Respuesta:'''


def prompt_estructurado(pregunta, contextos):
    return f'''Eres un asistente que responde preguntas basandote ESTRICTAMENTE en el contexto proporcionado, sin usar conocimiento externo. Sigue estos pasos antes de responder:

1. Identifica que fragmentos del contexto son relevantes para la pregunta.
2. Verifica si, con esos fragmentos, la pregunta se puede responder de forma completa, parcial, o no se puede responder.
3. Redacta la respuesta final basandote unicamente en lo que identificaste en el paso 1.

Contexto:
{formatear_contexto(contextos)}

Pregunta: {pregunta}

Responde siguiendo EXACTAMENTE este formato:
Fragmentos relevantes: <cuales y por que, brevemente>
Cobertura: <completa / parcial / nula>
Respuesta final: <tu respuesta aqui; si la cobertura es nula, escribe "No se encuentra en el contexto">'''


# --- formato RAFT (copia de raft_formato.py) ---
RAFT_MARCA_RESPUESTA = "##Answer:"
RAFT_SYSTEM_PROMPT = (
    "Eres un asistente que responde preguntas sobre el informe de autoevaluación "
    "con fines de acreditación del programa de Ingeniería de Sistemas. "
    "Recibirás varios documentos recuperados; algunos pueden no ser relevantes para la pregunta. "
    "Usa solo la información pertinente: razona paso a paso, copia textualmente los fragmentos "
    "que sustentan tu respuesta entre ##begin_quote## y ##end_quote##, y termina con la respuesta "
    "final después de '##Answer:'. Responde siempre en español."
)


def mensajes_raft(pregunta, contextos):
    documentos = "\n\n".join(f"<DOCUMENTO>\n{d.strip()}\n</DOCUMENTO>" for d in contextos)
    return [
        {"role": "system", "content": RAFT_SYSTEM_PROMPT},
        {"role": "user", "content": f"{documentos}\n\nPregunta: {pregunta.strip()}"},
    ]


# Cada estilo devuelve la lista de mensajes. directo/estructurado: un solo mensaje "user", como en el original.
ESTILOS_PROMPT = {
    "raft": mensajes_raft,
    "estructurado": lambda p, c: [{"role": "user", "content": prompt_estructurado(p, c)}],
    "directo": lambda p, c: [{"role": "user", "content": prompt_directo(p, c)}],
}


def extraer_respuesta_final(texto_generado, nombre_prompt):
    if texto_generado is None:
        return None
    if nombre_prompt == "estructurado":
        m = re.search(r"Respuesta final:\s*(.*)", texto_generado, re.DOTALL)
        return m.group(1).strip() if m else texto_generado.strip()
    if nombre_prompt == "raft":
        if RAFT_MARCA_RESPUESTA in texto_generado:
            return texto_generado.split(RAFT_MARCA_RESPUESTA, 1)[1].strip()
        return texto_generado.strip()
    return texto_generado.strip()

## 5. Cliente Ollama con reintentos (sin fallback de modelo)

Se usa la API nativa de Ollama (`/api/chat`) en lugar de la compatible con OpenAI por dos razones:
- permite fijar `num_ctx` en cada llamada;
- devuelve los conteos de tokens y los tiempos de cada etapa, utiles para el analisis de latencias.

In [5]:
def llamar_ollama(ollama_id, mensajes, intentos=5, espera_inicial=5):
    ultimo_error = None
    for intento in range(1, intentos + 1):
        try:
            t0 = time.perf_counter()
            r = _http.post("/api/chat", json={
                "model": ollama_id,
                "messages": mensajes,
                "stream": False,
                "options": {
                    "temperature": 0,
                    "seed": SEMILLA,
                    "num_ctx": NUM_CTX,
                    "num_predict": MAX_TOKENS_RESPUESTA,
                },
            })
            r.raise_for_status()
            d = r.json()
            tokens_prompt = d.get("prompt_eval_count") or 0
            tokens_generados = d.get("eval_count") or 0
            return {
                "texto": d["message"]["content"],
                "latencia_s": round(time.perf_counter() - t0, 3),
                "latencia_prompt_s": round((d.get("prompt_eval_duration") or 0) / 1e9, 3),
                "latencia_generacion_s": round((d.get("eval_duration") or 0) / 1e9, 3),
                "tokens_prompt": tokens_prompt,
                "tokens_generados": tokens_generados,
                # Si el prompt llena el contexto, Ollama pudo haber recortado el inicio
                "contexto_posiblemente_truncado": tokens_prompt >= NUM_CTX - 8,
                "respuesta_cortada_por_longitud": d.get("done_reason") == "length",
            }
        except Exception as e:
            ultimo_error = e
            espera = espera_inicial * (2 ** (intento - 1))
            print(f"    intento {intento}/{intentos} fallo ({e}); esperando {espera}s...")
            time.sleep(espera)
    print(f"    se agotaron los reintentos para {ollama_id}: {ultimo_error}")
    return None

## 6. Ejemplo rapido y estimacion de tiempo

La primera llamada carga el modelo en la GPU y tarda mas. Despues de esta celda, ejecuta `ollama ps` en una terminal: la columna **PROCESSOR** dice que porcentaje quedo en GPU. Con `100% GPU` es lo ideal. Con un reparto CPU/GPU funciona, pero mas lento.

In [6]:
nombre_modelo, info = next(iter(MODELOS_LOCALES.items()))
ejemplo = resultados_openai[0]
print("Pregunta:", ejemplo["pregunta"])
print("Esperada:", ejemplo.get("respuesta_esperada"), "\n")

_ = llamar_ollama(info["ollama_id"], [{"role": "user", "content": "Hola"}])   # calentamiento (carga del modelo)

tiempos = []
for nombre_prompt in ESTILOS_A_CORRER:
    res = llamar_ollama(info["ollama_id"], ESTILOS_PROMPT[nombre_prompt](ejemplo["pregunta"], ejemplo["contextos_recuperados"]))
    tiempos.append(res["latencia_s"])
    print(f"=== {nombre_prompt}  ({res['latencia_s']} s, prompt={res['tokens_prompt']} tok, salida={res['tokens_generados']} tok)")
    print(res["texto"][:1200])
    print(f"--> respuesta_generada: {extraer_respuesta_final(res['texto'], nombre_prompt)[:300]}\n")

n_comb = len(PIPELINES_RETRIEVE) * len(MODELOS_LOCALES) * len(ESTILOS_A_CORRER)
n_preg = TAMANO_MUESTRA_LLM or len(resultados_openai)
print(f"Estimacion corrida completa: ~{sum(tiempos)/len(tiempos) * n_comb * n_preg / 3600:.1f} h "
      f"({n_comb} combinaciones x {n_preg} preguntas)")

Pregunta: Compare la duración en semestres y el total de créditos de los programas de Ingeniería de Sistemas de la Universidad de Antioquia y la Universidad Nacional de Colombia según la Tabla 1.3.
Esperada: La Universidad de Antioquia tiene un programa de Ingeniería de Sistemas con duración de 10 semestres y 180 créditos totales, mientras que la Universidad Nacional de Colombia tiene un programa con duración de 10 semestres y 165 créditos totales. 

=== raft  (26.43 s, prompt=2484 tok, salida=411 tok)
Para responder a la pregunta, se deben comparar los datos de los programas de Ingeniería de Sistemas de la Universidad de Antioquia y la Universidad Nacional de Colombia según la Tabla 1.3.

La Tabla 1.3 no se proporciona en la pregunta, pero se puede inferir que se refiere a la Tabla 1.3 del documento original, que es la siguiente:

|**Universidad**|**Sede**|**Nombre del Programa**<br>**Académico**|**Modalidad -**<br>**Jornada**|**Duración**<br>**semestres**|**Total**<br>**créditos**|
|

## 7. Generacion completa (3 combinaciones, con checkpoint)

Funciona como el notebook original: cada combinacion se guarda en su checkpoint al terminar, y si existe se reutiliza. Como el modelo local es mas lento, ademas se guarda un **checkpoint parcial cada 10 preguntas** (`*.parcial.json`). Si la laptop se suspende o se interrumpe la celda, al volver a ejecutarla continua donde iba.

Para que no se interrumpa: conecta la laptop a la corriente y desactiva la suspension mientras corre.

In [7]:
def generar_para_combinacion(nombre_pipeline, resultados_pipeline, nombre_modelo, ollama_id, nombre_prompt, funcion_prompt):
    etiqueta = f"{nombre_pipeline}__{nombre_modelo}__{nombre_prompt}"
    ruta_checkpoint = RUTA_CHECKPOINTS_LLM / f"{etiqueta}.json"
    ruta_parcial = RUTA_CHECKPOINTS_LLM / f"{etiqueta}.parcial.json"

    if ruta_checkpoint.exists():
        print(f"  [{etiqueta}] ya generado antes, se reutiliza el checkpoint.")
        with open(ruta_checkpoint, "r", encoding="utf-8") as f:
            return json.load(f)

    preguntas_a_usar = resultados_pipeline[:TAMANO_MUESTRA_LLM] if TAMANO_MUESTRA_LLM else resultados_pipeline

    salida = []
    if ruta_parcial.exists():
        with open(ruta_parcial, "r", encoding="utf-8") as f:
            salida = json.load(f)
        print(f"  [{etiqueta}] retomando desde la pregunta {len(salida) + 1}")
    hechas = {r["id"] for r in salida}

    for i, r in enumerate(preguntas_a_usar, 1):
        if r["id"] in hechas:
            continue
        res = llamar_ollama(ollama_id, funcion_prompt(r["pregunta"], r["contextos_recuperados"]))
        respuesta_cruda = res["texto"] if res else None
        salida.append({
            "id": r["id"],
            "pregunta": r["pregunta"],
            "respuesta_esperada": r.get("respuesta_esperada"),
            "evidencia_gold": r.get("evidencia_gold"),
            "tipo": r.get("tipo"),
            "dificultad": r.get("dificultad"),
            "tipo_evidencia": r.get("tipo_evidencia"),
            "contextos_usados": r["contextos_recuperados"],
            "respuesta_generada_cruda": respuesta_cruda,
            "respuesta_generada": extraer_respuesta_final(respuesta_cruda, nombre_prompt),
            # campos adicionales (no existen en las combinaciones de OpenRouter)
            "latencia_s": res["latencia_s"] if res else None,
            "latencia_prompt_s": res["latencia_prompt_s"] if res else None,
            "latencia_generacion_s": res["latencia_generacion_s"] if res else None,
            "tokens_prompt": res["tokens_prompt"] if res else None,
            "tokens_generados": res["tokens_generados"] if res else None,
            "contexto_posiblemente_truncado": res["contexto_posiblemente_truncado"] if res else None,
            "respuesta_cortada_por_longitud": res["respuesta_cortada_por_longitud"] if res else None,
        })
        if len(salida) % 10 == 0:
            with open(ruta_parcial, "w", encoding="utf-8") as f:
                json.dump(salida, f, ensure_ascii=False)
        if i % 25 == 0 or i == len(preguntas_a_usar):
            print(f"    [{etiqueta}] {i}/{len(preguntas_a_usar)}")

    # mismo orden que el original (por pregunta)
    orden = {r["id"]: k for k, r in enumerate(preguntas_a_usar)}
    salida.sort(key=lambda x: orden.get(x["id"], 10**9))

    with open(ruta_checkpoint, "w", encoding="utf-8") as f:
        json.dump(salida, f, ensure_ascii=False, indent=2)
    if ruta_parcial.exists():
        ruta_parcial.unlink()
    print(f"  [{etiqueta}] listo, {len(salida)} respuestas guardadas en {ruta_checkpoint}")
    return salida

In [8]:
combinaciones_resultado = []

for nombre_pipeline, resultados_pipeline in PIPELINES_RETRIEVE.items():
    for nombre_modelo, info_modelo in MODELOS_LOCALES.items():
        for nombre_prompt in ESTILOS_A_CORRER:
            print(f"\n=== {nombre_pipeline} | {nombre_modelo} | prompt={nombre_prompt} ===")
            resultados_generados = generar_para_combinacion(
                nombre_pipeline, resultados_pipeline, nombre_modelo,
                info_modelo["ollama_id"], nombre_prompt, ESTILOS_PROMPT[nombre_prompt],
            )
            combinaciones_resultado.append({
                "pipeline_retrieve": nombre_pipeline,
                "modelo_llm": nombre_modelo,
                "openrouter_id": f"ollama/{info_modelo['ollama_id']}",   # mismo campo que el original; aqui es local
                "rol_modelo": info_modelo["rol"],
                "estilo_prompt": nombre_prompt,
                "total_preguntas": len(resultados_generados),
                "resultados": resultados_generados,
            })

salida_final = {
    "dataset_fuente": "dataset_validacion_final.json",
    "criterio_filtro": "preguntas con llm_context_precision_with_reference > 0 y context_recall > 0 en AMBOS pipelines de retrieve (openai cs800 denso, BGE-m3 hibrido cs800)",
    "total_preguntas_filtradas": len(resultados_openai),
    "generacion": {
        "motor": "ollama (local)",
        "modelo": "Llama-3.1-8B-Instruct sin afinar (control), GGUF Q4_K_M (ollama: llama3.1:8b-instruct-q4_K_M)",
        "num_ctx": NUM_CTX, "max_tokens": MAX_TOKENS_RESPUESTA, "temperature": 0, "seed": SEMILLA,
        "contextos": "solo openai_cs800 (text-embedding-3-large denso, cs800, k=5)",
    },
    "combinaciones": combinaciones_resultado,
}

with open(RUTA_SALIDA_LLM, "w", encoding="utf-8") as f:
    json.dump(salida_final, f, ensure_ascii=False, indent=2)
print(f"\nListo. {len(combinaciones_resultado)} combinaciones guardadas en {RUTA_SALIDA_LLM}")


=== openai_cs800 | llama3.1-8b-base | prompt=raft ===
    [openai_cs800__llama3.1-8b-base__raft] 25/196
    [openai_cs800__llama3.1-8b-base__raft] 50/196
    [openai_cs800__llama3.1-8b-base__raft] 75/196
    [openai_cs800__llama3.1-8b-base__raft] 100/196
    [openai_cs800__llama3.1-8b-base__raft] 125/196
    [openai_cs800__llama3.1-8b-base__raft] 150/196
    [openai_cs800__llama3.1-8b-base__raft] 175/196
    [openai_cs800__llama3.1-8b-base__raft] 196/196
  [openai_cs800__llama3.1-8b-base__raft] listo, 196 respuestas guardadas en validacion\checkpoints_llm\openai_cs800__llama3.1-8b-base__raft.json

=== openai_cs800 | llama3.1-8b-base | prompt=estructurado ===
    [openai_cs800__llama3.1-8b-base__estructurado] 25/196
    [openai_cs800__llama3.1-8b-base__estructurado] 50/196
    [openai_cs800__llama3.1-8b-base__estructurado] 75/196
    [openai_cs800__llama3.1-8b-base__estructurado] 100/196
    [openai_cs800__llama3.1-8b-base__estructurado] 125/196
    [openai_cs800__llama3.1-8b-base__est

## 8. Verificacion

In [9]:
tamano_mb = Path(RUTA_SALIDA_LLM).stat().st_size / (1024 * 1024)
print(f"Archivo: {RUTA_SALIDA_LLM} ({tamano_mb:.2f} MB)\n")

filas = []
for c in combinaciones_resultado:
    rs = c["resultados"]
    lat = [r["latencia_s"] for r in rs if r.get("latencia_s") is not None]
    filas.append({
        "combinacion": f"{c['pipeline_retrieve']}__{c['modelo_llm']}__{c['estilo_prompt']}",
        "con_respuesta": sum(1 for r in rs if r["respuesta_generada"]),
        "total": c["total_preguntas"],
        "raft_con_##Answer": sum(1 for r in rs if (r["respuesta_generada_cruda"] or "").count("##Answer:")) if c["estilo_prompt"] == "raft" else None,
        "estructurado_con_formato": sum(1 for r in rs if "Respuesta final:" in (r["respuesta_generada_cruda"] or "")) if c["estilo_prompt"] == "estructurado" else None,
        "contexto_truncado": sum(1 for r in rs if r.get("contexto_posiblemente_truncado")),
        "respuesta_cortada": sum(1 for r in rs if r.get("respuesta_cortada_por_longitud")),
        "latencia_media_s": round(sum(lat) / len(lat), 2) if lat else None,
        "tokens_prompt_medio": round(sum(r["tokens_prompt"] or 0 for r in rs) / max(1, len(rs))),
    })
pd.DataFrame(filas)

Archivo: ./validacion/resultados_generacion_llm_base.json (8.00 MB)



,combinacion,con_respuesta,total,raft_con_##Answer,estructurado_con_formato,contexto_truncado,respuesta_cortada,latencia_media_s,tokens_prompt_medio
0,openai_cs800__llama3.1-8b-base__raft,196,196,163.0,NaN,0,18,29.16,3183
1,openai_cs800__llama3.1-8b-base__estructurado,196,196,NaN,195.0,0,2,12.50,3238
2,openai_cs800__llama3.1-8b-base__directo,196,196,NaN,NaN,0,0,8.54,3103


**Como leer la tabla:**
- `contexto_truncado` debe ser 0.
- `raft_con_##Answer` y `estructurado_con_formato` muestran si el modelo base sigue el formato pedido solo con las instrucciones. Es parte de lo que se compara con el modelo afinado.

**Siguiente paso:** `evaluacion_generacion_ragas_base.ipynb`.